# Task 46: Cost-Sensitive Model Selection

Balance class representation through manual resampling, choose the model and probability threshold on validation precision/recall/F1 utility, and report the selected model once on untouched test data.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

candidates=[Path('heart.csv'),Path('Task_46_Cost_Sensitive_Model_Selection/heart.csv'),Path('Heart_Disease_Tasks/Task_46_Cost_Sensitive_Model_Selection/heart.csv')]
data=pd.read_csv(next(p for p in candidates if p.is_file())).drop_duplicates().reset_index(drop=True)
features=[c for c in ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal'] if c in data]
X,y=data[features].to_numpy(float),data.target.to_numpy(int)

def split(labels,fraction,seed):
    rng=np.random.default_rng(seed);a=[];b=[]
    for label in np.unique(labels):
        ids=np.flatnonzero(labels==label);rng.shuffle(ids);cut=round(fraction*len(ids));a.extend(ids[:cut]);b.extend(ids[cut:])
    rng.shuffle(a);rng.shuffle(b);return np.array(a),np.array(b)

trainval,test=split(y,.8,46);train_local,val_local=split(y[trainval],.75,47);train,val=trainval[train_local],trainval[val_local]
Xtr,Xv,Xte=X[train],X[val],X[test];ytr,yv,yte=y[train],y[val],y[test]
mean,std=Xtr.mean(0),Xtr.std(0);std[std==0]=1
Xtr,Xv,Xte=(Xtr-mean)/std,(Xv-mean)/std,(Xte-mean)/std

# Manual random oversampling gives every observed class the same training count.
rng=np.random.default_rng(48);sampled=[]
for label in np.unique(ytr):
    ids=np.flatnonzero(ytr==label);sampled.extend(rng.choice(ids,size=max(np.bincount(ytr)),replace=len(ids)<max(np.bincount(ytr))))
sampled=np.array(sampled);rng.shuffle(sampled);Xbalanced,ybalanced=Xtr[sampled],ytr[sampled]
print('Original training counts:',dict(zip(*np.unique(ytr,return_counts=True))))
print('Balanced training counts:',dict(zip(*np.unique(ybalanced,return_counts=True))))


def sigmoid(z):return 1/(1+np.exp(-np.clip(z,-35,35)))
class Logistic:
    def fit(self,x,y):
        self.w=np.zeros(x.shape[1]);self.b=0.
        for _ in range(1200):
            p=sigmoid(x@self.w+self.b);e=p-y;self.w-=.06*(x.T@e/len(y)+.02*self.w);self.b-=.06*e.mean()
        return self
    def proba(self,x):return sigmoid(x@self.w+self.b)
class DecisionStump:
    def fit(self,x,y):
        parent=1-np.sum((np.bincount(y,minlength=2)/len(y))**2);best=None
        for j in range(x.shape[1]):
            values=np.unique(x[:,j])
            for threshold in (values[:-1]+values[1:])/2:
                mask=x[:,j]<=threshold
                if not mask.any() or mask.all():continue
                left=np.bincount(y[mask],minlength=2)/mask.sum();right=np.bincount(y[~mask],minlength=2)/(~mask).sum()
                gain=parent-mask.mean()*(1-np.sum(left**2))-(~mask).mean()*(1-np.sum(right**2))
                if best is None or gain>best[0]:best=(gain,j,threshold,mask)
        if best is None: self.j,self.threshold,self.left_prob=0,0,float(y.mean());self.right_prob=self.left_prob
        else:
            _,self.j,self.threshold,mask=best;self.left_prob=float(y[mask].mean());self.right_prob=float(y[~mask].mean())
        return self
    def proba(self,x):return np.where(x[:,self.j]<=self.threshold,self.left_prob,self.right_prob)
class KNN:
    def __init__(self,k=9):self.k=k
    def fit(self,x,y):self.x,self.y=x,y;return self
    def proba(self,x):
        result=[]
        for row in x:
            ids=np.argsort(np.sum((self.x-row)**2,axis=1),kind='stable')[:self.k];result.append(float(self.y[ids].mean()))
        return np.array(result)


def calculate(actual,pred):
    tp=np.sum((actual==1)&(pred==1));tn=np.sum((actual==0)&(pred==0));fp=np.sum((actual==0)&(pred==1));fn=np.sum((actual==1)&(pred==0))
    precision=tp/(tp+fp) if tp+fp else 0.;recall=tp/(tp+fn) if tp+fn else 0.
    return {'accuracy':float((tp+tn)/len(actual)),'precision':float(precision),'recall':float(recall),'f1':float(2*precision*recall/(precision+recall)) if precision+recall else 0.}

# A recall-forward utility reflects the higher cost assigned to missed positive cases in this demonstration.
weights={'precision':.2,'recall':.3,'f1':.5};thresholds=[.25,.35,.45,.50,.55,.65,.75]
models={'Logistic regression':Logistic().fit(Xbalanced,ybalanced),'Decision stump':DecisionStump().fit(Xbalanced,ybalanced),'k-NN k=9':KNN(9).fit(Xbalanced,ybalanced)}
validation_rows=[];chosen={}
for name,model in models.items():
    probabilities=model.proba(Xv);best=None
    for threshold in thresholds:
        report=calculate(yv,(probabilities>=threshold).astype(int));utility=sum(weights[key]*report[key] for key in weights)
        if best is None or utility>best[0]:best=(utility,threshold,report)
    chosen[name]={'utility':best[0],'threshold':best[1],**best[2]};validation_rows.append({'Model':name,**chosen[name]})
validation_table=pd.DataFrame(validation_rows).set_index('Model')
best_name=validation_table['utility'].idxmax();best_threshold=chosen[best_name]['threshold']
test_probability=models[best_name].proba(Xte);test_report=calculate(yte,(test_probability>=best_threshold).astype(int))
print('Validation selection weights:',weights);display(validation_table.round(3))
print('Selected model:',best_name,'threshold:',best_threshold)
print('Untouched test metrics:',test_report)
print('Precision/recall trade-off: lowering the threshold often increases recall while reducing precision. Resampling may improve minority recall, but deployment prevalence and error costs should determine the final policy.')